# AI-Driven Cloud Auto-Scaling — Model Training Notebook

This notebook walks through the complete ML pipeline:
1. Load the **15-day hourly** workload dataset (360 records)
2. Explore & visualise the data (daily cycles, weekly patterns)
3. Preprocess & engineer time-aware features
4. Train a Linear Regression model
5. Evaluate performance (R², MAE)
6. Save the trained model
7. Test predictions on sample inputs

---
## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import math
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries imported successfully!")

---
## 2. Load the Dataset

In [ ]:
# Load the workload dataset
DATASET_PATH = os.path.join("..", "dataset", "workload_dataset.csv")
df = pd.read_csv(DATASET_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])

print(f"Dataset shape: {df.shape}  (15 days x 24 hours = 360 records)")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
# Basic statistics
print("Dataset Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Plot all metrics over 15 days
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df['datetime'], df['requests'], color='#2196F3', linewidth=1)
axes[0].set_ylabel('Requests / hour')
axes[0].set_title('Workload Traffic Pattern Over 15 Days')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['datetime'], df['cpu_usage'], color='#FF5722', linewidth=1)
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='SLA Threshold (80%)')
axes[1].set_ylabel('CPU Usage (%)')
axes[1].set_title('CPU Utilisation Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['datetime'], df['memory_usage'], color='#4CAF50', linewidth=1)
axes[2].set_ylabel('Memory Usage (%)')
axes[2].set_xlabel('Date & Time')
axes[2].set_title('Memory Utilisation Over Time')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Average traffic by hour of day
hourly_avg = df.groupby('hour')['requests'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hourly_avg.index, hourly_avg.values, color='#FF9800', edgecolor='white')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Requests')
ax.set_title('Average Hourly Traffic Pattern (Business Hours Peak)')
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print("Key insight: traffic peaks during 10:00-14:00 (business hours)")
print("and drops to minimum during 00:00-05:00 (night).")

In [ ]:
# Weekday vs Weekend comparison
weekday_avg = df[df['is_weekend'] == 0].groupby('hour')['requests'].mean()
weekend_avg = df[df['is_weekend'] == 1].groupby('hour')['requests'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weekday_avg.index, weekday_avg.values, 'o-', label='Weekday', color='#2196F3', linewidth=2)
ax.plot(weekend_avg.index, weekend_avg.values, 's-', label='Weekend', color='#FF5722', linewidth=2)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Requests')
ax.set_title('Weekday vs Weekend Traffic Pattern')
ax.set_xticks(range(24))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Weekday avg: {weekday_avg.mean():.0f} reqs/hour")
print(f"Weekend avg: {weekend_avg.mean():.0f} reqs/hour")
print(f"Weekend traffic is ~{(1 - weekend_avg.mean()/weekday_avg.mean())*100:.0f}% lower")

In [ ]:
# Correlation heatmap
corr_cols = ['hour', 'day_of_week', 'is_weekend', 'requests', 'cpu_usage', 'memory_usage']
correlation = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(correlation, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45)
ax.set_yticklabels(corr_cols)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{correlation.iloc[i, j]:.3f}',
                ha='center', va='center', fontweight='bold', fontsize=9)

plt.colorbar(im)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing & Feature Engineering

In [ ]:
# Step 1: Handle missing values
df_clean = df.ffill().dropna().reset_index(drop=True)
print(f"After cleaning - shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

In [ ]:
# Step 2: Feature Engineering
# Features: [hour, day_of_week, is_weekend, requests, cpu_usage]
# Target:   next row's [requests] (shifted by -1)

FEATURE_COLUMNS = ['hour', 'day_of_week', 'is_weekend', 'requests', 'cpu_usage']
TARGET_COLUMN = 'requests'

df_clean['target'] = df_clean[TARGET_COLUMN].shift(-1)
df_clean = df_clean.dropna(subset=['target'])

print(f"After feature engineering - shape: {df_clean.shape}")
print(f"\nFeatures used: {FEATURE_COLUMNS}")
print(f"Target: next-step request count")
print(f"\nSample rows:")
df_clean[['datetime'] + FEATURE_COLUMNS + ['target']].head(10)

In [ ]:
# Step 3: Separate features (X) and target (y)
X = df_clean[FEATURE_COLUMNS].values
y = df_clean['target'].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:  {y.shape}")
print(f"\nFeatures: {FEATURE_COLUMNS}")
print(f"  - hour:         captures daily traffic cycle (0-23)")
print(f"  - day_of_week:  captures weekly patterns (0=Sun, 6=Sat)")
print(f"  - is_weekend:   binary flag for weekend traffic reduction")
print(f"  - requests:     current workload level")
print(f"  - cpu_usage:    current resource utilisation")

---
## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Split ratio:  {X_train.shape[0]}/{X_test.shape[0]} = 80%/20%")

---
## 6. Train the Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel coefficients:")
for feature, coef in zip(FEATURE_COLUMNS, model.coef_):
    print(f"  {feature:>15}: {coef:.6f}")
print(f"  {'intercept':>15}: {model.intercept_:.6f}")
print(f"\nPrediction formula:")
terms = ' + '.join([f'({c:.4f} x {f})' for f, c in zip(FEATURE_COLUMNS, model.coef_)])
print(f"  predicted_requests = {terms} + ({model.intercept_:.4f})")

---
## 7. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("       MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"  R-squared (R2):              {r2:.4f}")
print(f"  Mean Absolute Error (MAE):   {mae:.4f}")
print(f"  Mean Squared Error (MSE):    {mse:.4f}")
print(f"  Root Mean Squared Error:     {rmse:.4f}")
print("=" * 50)
print(f"\nInterpretation:")
print(f"  - R2 = {r2:.4f} means the model explains {r2*100:.1f}% of the variance")
print(f"  - MAE = {mae:.1f} means predictions are off by ~{mae:.0f} requests on average")

In [ ]:
# Actual vs Predicted scatter + residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.7, color='#2196F3', edgecolors='white')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Requests')
axes[0].set_ylabel('Predicted Requests')
axes[0].set_title(f'Actual vs Predicted (R2 = {r2:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.7, color='#FF5722', edgecolors='white')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].set_xlabel('Predicted Requests')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residuals Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(residuals, bins=20, color='#9C27B0', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Residual (Actual - Predicted)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Prediction Errors')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.4f} (should be close to 0)")
print(f"Std of residuals: {residuals.std():.4f}")

---
## 8. Save the Trained Model

In [ ]:
MODEL_DIR = os.path.join("..", "models")
MODEL_PATH = os.path.join(MODEL_DIR, "trained_model.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, MODEL_PATH)

print(f"Model saved to: {os.path.abspath(MODEL_PATH)}")
print(f"File size: {os.path.getsize(MODEL_PATH)} bytes")

In [ ]:
# Verify reload
loaded_model = joblib.load(MODEL_PATH)
y_verify = loaded_model.predict(X_test)
print(f"Reloaded model R2: {r2_score(y_test, y_verify):.4f}")
print(f"Match: {np.allclose(y_pred, y_verify)}")

---
## 9. Test Predictions on Sample Scenarios

In [ ]:
# Test scenarios with realistic datetime context
test_scenarios = [
    {"name": "Night (3AM Mon)",    "hour": 3,  "day_of_week": 1, "is_weekend": 0, "requests": 65,  "cpu_usage": 15.0},
    {"name": "Morning (9AM Tue)",  "hour": 9,  "day_of_week": 2, "is_weekend": 0, "requests": 250, "cpu_usage": 55.0},
    {"name": "Peak (12PM Wed)",    "hour": 12, "day_of_week": 3, "is_weekend": 0, "requests": 400, "cpu_usage": 85.0},
    {"name": "Evening (20PM Thu)", "hour": 20, "day_of_week": 4, "is_weekend": 0, "requests": 180, "cpu_usage": 40.0},
    {"name": "Weekend (12PM Sat)", "hour": 12, "day_of_week": 6, "is_weekend": 1, "requests": 280, "cpu_usage": 60.0},
]

print(f"{'Scenario':<22} {'Hour':>4} {'Day':>4} {'Wknd':>5} {'CurrReqs':>8} {'CPU':>5} {'Predicted':>10} {'Containers':>11}")
print("-" * 80)

for s in test_scenarios:
    features = np.array([[s['hour'], s['day_of_week'], s['is_weekend'], s['requests'], s['cpu_usage']]])
    prediction = max(0, int(round(loaded_model.predict(features)[0])))
    containers = math.ceil(prediction / 100)
    day_names = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
    print(f"{s['name']:<22} {s['hour']:>4} {day_names[s['day_of_week']]:>4} {'Y' if s['is_weekend'] else 'N':>5} {s['requests']:>8} {s['cpu_usage']:>5.1f} {prediction:>10} {containers:>11}")

In [ ]:
# Full dataset prediction vs actual
df_full = pd.read_csv(DATASET_PATH)
df_full['datetime'] = pd.to_datetime(df_full['datetime'])
df_full['target'] = df_full['requests'].shift(-1)
df_full = df_full.dropna(subset=['target'])

X_full = df_full[FEATURE_COLUMNS].values
y_full = df_full['target'].values
y_full_pred = loaded_model.predict(X_full)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_full['datetime'], y_full, label='Actual Next Requests',
        linewidth=1.5, color='#2196F3')
ax.plot(df_full['datetime'], y_full_pred, label='Predicted Next Requests',
        linewidth=1.5, linestyle='--', color='#FF5722')
ax.set_xlabel('Date & Time')
ax.set_ylabel('Requests / hour')
ax.set_title('Full Dataset: Actual vs Predicted Workload (15 Days)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

full_r2 = r2_score(y_full, y_full_pred)
full_mae = mean_absolute_error(y_full, y_full_pred)
print(f"Full dataset R2:  {full_r2:.4f}")
print(f"Full dataset MAE: {full_mae:.4f}")

---
## 10. Summary

| Item | Value |
|------|-------|
| **Dataset** | 15 days x 24 hours = 360 hourly records |
| **Model** | Linear Regression (scikit-learn) |
| **Features** | hour, day_of_week, is_weekend, requests, cpu_usage |
| **Target** | Next hour's request count |
| **Train/Test Split** | 80% / 20% |
| **R-squared** | ~0.90 |
| **MAE** | ~29.5 |
| **Model File** | `models/trained_model.pkl` |

The model uses **time-aware features** (hour of day, day of week, weekend flag) to capture realistic daily and weekly traffic patterns.